# NLP Assignment PS6 S2 2025 — Corrected Final
## Reading Comprehension Question Answering using an Attention-based Encoder-Decoder

**Dataset:** SQuAD v1.1 (`train-v1.1.json`)

This notebook is aligned with all three tasks in the assignment:
- Task 1: nested JSON parsing, preprocessing, length analysis, tokenization and padding.
- Task 2: attention-based Encoder-Decoder architecture, 15-epoch training, curves, EM/F1 evaluation and five predictions.
- Task 3: evidence-based analysis of two failed predictions and architectural refinement.

**Important:** Run this notebook from top to bottom in **Jupyter Notebook**. The numerical metrics, prediction examples, and error diagnoses are generated by the actual model run and should be retained when exporting the executed notebook to PDF.

> **Runtime note:** `MAX_SAMPLES = 20,000` is used by default because this generative Seq2Seq model is computationally expensive. The complete pipeline remains compatible with the full SQuAD dataset; set `MAX_SAMPLES = len(df)` if your hardware can support the full experiment.


## 1. Setup
Place `train-v1.1.json` in the same folder as this notebook, or update `DATA_PATH` below. Install missing packages using `pip install tensorflow pandas scikit-learn matplotlib nltk`.

In [ ]:
import json
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM, Dense, Concatenate,
    AdditiveAttention, TimeDistributed
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

DATA_PATH = "train-v1.1.json"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

MAX_SAMPLES = 20000
# Assignment requirement: train for a minimum of 15–20 epochs.
# We use exactly 15 epochs and do NOT use early stopping, so the minimum is guaranteed.
EPOCHS = 15
BATCH_SIZE = 64
EMBEDDING_DIM = 128
LATENT_DIM = 128

print("TensorFlow version:", tf.__version__)
print("Dataset path:", DATA_PATH)

## Task 1 — Parsing Nested Data and Sequence Preprocessing

### 1.1 Parse the nested SQuAD JSON
Each SQuAD article contains paragraphs; each paragraph contains multiple question-answer entries. We flatten it into one row per question-answer pair using the first annotated answer.

**Observation:** Flattening preserves the relationship between a context, its question, and an extractive ground-truth answer. This is the appropriate tabular format for model training.

In [ ]:
if not Path(DATA_PATH).exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Download SQuAD v1.1 and place train-v1.1.json beside this notebook."
    )

with open(DATA_PATH, "r", encoding="utf-8") as f:
    squad = json.load(f)

records = []
for article in squad["data"]:
    title = article.get("title", "")
    for paragraph in article.get("paragraphs", []):
        context = paragraph.get("context", "")
        for qa in paragraph.get("qas", []):
            answers = qa.get("answers", [])
            if answers:
                records.append({
                    "title": title,
                    "context": context,
                    "question": qa.get("question", ""),
                    "answer_text": answers[0].get("text", ""),
                    "answer_start": answers[0].get("answer_start", -1)
                })

df = pd.DataFrame(records)
print("Flattened DataFrame shape:", df.shape)
display(df.head())

### 1.2 Clean the text
Text is lowercased, excess whitespace is removed, and sentence-ending punctuation (`.`, `?`, `!`) is retained.

**Observation:** Cleaning reduces sparsity in the vocabulary while retaining punctuation that can provide sentence-boundary information.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9.,!?';:\-\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["context_clean"] = df["context"].map(clean_text)
df["question_clean"] = df["question"].map(clean_text)
df["answer_clean"] = df["answer_text"].map(clean_text)

display(df[["context_clean", "question_clean", "answer_clean"]].head())

### 1.3 Analyze token-length distributions
For this notebook, sequence length means whitespace-token count. The 95th percentile is used as the padding cutoff, balancing memory cost against context retention.

**Inference:** Contexts are expected to be substantially longer than questions and answers. Very long contexts may still be truncated, which can lead to answer errors when the relevant span falls beyond the cutoff.

In [ ]:
for col in ["context_clean", "question_clean", "answer_clean"]:
    df[col.replace("_clean", "_len")] = df[col].str.split().str.len()

length_cols = ["context_len", "question_len", "answer_len"]
display(df[length_cols].describe().T)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, length_cols):
    ax.hist(df[col], bins=50, color="steelblue", edgecolor="white")
    ax.set_title(f"{col.replace('_', ' ').title()} Distribution")
    ax.set_xlabel("Tokens")
    ax.set_ylabel("Frequency")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "length_distributions.png", dpi=150)
plt.show()

### 1.4 Create tokenizers and padded sequences
A shared encoder tokenizer is fitted on contexts and questions; a separate decoder tokenizer is fitted on answers. The decoder uses `<start>` and `<end>` markers for teacher forcing.

**Observation:** A shared input vocabulary lets the encoder represent words occurring in both questions and contexts. A decoder vocabulary focuses model capacity on answer generation.

In [ ]:
context_maxlen = int(df["context_len"].quantile(0.95))
question_maxlen = int(df["question_len"].quantile(0.95))
answer_maxlen = int(df["answer_len"].quantile(0.95)) + 2

encoder_tokenizer = Tokenizer(oov_token="<oov>")
encoder_tokenizer.fit_on_texts(
    df["context_clean"].tolist() + df["question_clean"].tolist()
)

decoder_texts = [f"<start> {x} <end>" for x in df["answer_clean"]]
decoder_tokenizer = Tokenizer(oov_token="<oov>", filters='!"#$%&()*+,-./:;=?@[\]^_`{|}~\t\n')
decoder_tokenizer.fit_on_texts(decoder_texts)

encoder_vocab_size = len(encoder_tokenizer.word_index) + 1
decoder_vocab_size = len(decoder_tokenizer.word_index) + 1

context_seq = encoder_tokenizer.texts_to_sequences(df["context_clean"])
question_seq = encoder_tokenizer.texts_to_sequences(df["question_clean"])
answer_seq = decoder_tokenizer.texts_to_sequences(decoder_texts)

context_pad = pad_sequences(context_seq, maxlen=context_maxlen, padding="post", truncating="post")
question_pad = pad_sequences(question_seq, maxlen=question_maxlen, padding="post", truncating="post")
answer_pad = pad_sequences(answer_seq, maxlen=answer_maxlen, padding="post", truncating="post")

decoder_input_all = answer_pad[:, :-1]
decoder_target_all = answer_pad[:, 1:]

print(f"Encoder vocabulary size: {encoder_vocab_size:,}")
print(f"Decoder vocabulary size: {decoder_vocab_size:,}")
print(f"Context cutoff (95th percentile): {context_maxlen}")
print(f"Question cutoff (95th percentile): {question_maxlen}")
print(f"Answer cutoff including boundary markers: {answer_maxlen}")
print("Context padded shape:", context_pad.shape)
print("Question padded shape:", question_pad.shape)
print("Decoder input shape:", decoder_input_all.shape)
print("Decoder target shape:", decoder_target_all.shape)

# Explicit word-to-index mappings required by Task 1.
context_word_to_index = encoder_tokenizer.word_index.copy()
question_word_to_index = encoder_tokenizer.word_index.copy()  # shared encoder vocabulary
answer_word_to_index = decoder_tokenizer.word_index.copy()

print("\nWord-to-index mapping examples:")
print("Context:", list(context_word_to_index.items())[:10])
print("Question:", list(question_word_to_index.items())[:10])
print("Answer:", list(answer_word_to_index.items())[:10])


## Task 2 — Attention-Based Encoder-Decoder Model

### 2.1 Prepare train and validation data
A reproducible 80:20 holdout split is used. The optional sample limit is applied after preprocessing so the code remains compatible with the complete dataset.

**Inference:** A holdout validation set provides an unbiased estimate of generalization during training, while a fixed random seed makes results reproducible.

In [ ]:
n_samples = min(MAX_SAMPLES, len(df))
indices = np.arange(n_samples)

train_idx, val_idx = train_test_split(indices, test_size=0.20, random_state=SEED)

ctx_tr, ctx_val = context_pad[train_idx], context_pad[val_idx]
q_tr, q_val = question_pad[train_idx], question_pad[val_idx]
dec_in_tr, dec_in_val = decoder_input_all[train_idx], decoder_input_all[val_idx]
dec_tgt_tr, dec_tgt_val = decoder_target_all[train_idx], decoder_target_all[val_idx]

print("Training examples:", len(train_idx))
print("Validation examples:", len(val_idx))

# Padding positions should not contribute to the loss/accuracy.
# This prevents token accuracy from being artificially inflated by zero-padding.
train_sample_weights = (dec_tgt_tr != 0).astype("float32")
val_sample_weights = (dec_tgt_val != 0).astype("float32")

print("Non-padding target tokens in training:", int(train_sample_weights.sum()))
print("Non-padding target tokens in validation:", int(val_sample_weights.sum()))


### 2.2 Build the model
The context and question are encoded using a shared embedding layer and separate bidirectional LSTMs. Their token-level outputs are concatenated and passed to additive attention. The decoder predicts the next answer token at each time step.

**Observation:** Attention creates a weighted context vector for every decoder position, helping the decoder focus on relevant words from the context-question representation rather than relying only on a fixed-length vector.

In [ ]:
context_input = Input(shape=(context_maxlen,), name="context_input")
question_input = Input(shape=(question_maxlen,), name="question_input")
decoder_input = Input(shape=(answer_maxlen - 1,), name="decoder_input")

encoder_embedding = Embedding(encoder_vocab_size, EMBEDDING_DIM, mask_zero=True, name="encoder_embedding")
context_emb = encoder_embedding(context_input)
question_emb = encoder_embedding(question_input)

context_encoder = Bidirectional(LSTM(LATENT_DIM, return_sequences=True), name="context_bilstm")(context_emb)
question_encoder = Bidirectional(LSTM(LATENT_DIM, return_sequences=True), name="question_bilstm")(question_emb)
encoder_outputs = Concatenate(axis=1, name="context_question_states")([context_encoder, question_encoder])

# Mean pooling gives a fixed-size representation used to initialize both decoder states.
encoder_pool = tf.keras.layers.GlobalAveragePooling1D(name="encoder_pool")(encoder_outputs)
initial_h = Dense(2 * LATENT_DIM, activation="tanh", name="initial_h")(encoder_pool)
initial_c = Dense(2 * LATENT_DIM, activation="tanh", name="initial_c")(encoder_pool)

decoder_embedding = Embedding(decoder_vocab_size, EMBEDDING_DIM, mask_zero=True, name="decoder_embedding")
dec_emb = decoder_embedding(decoder_input)
decoder_outputs = LSTM(2 * LATENT_DIM, return_sequences=True, name="decoder_lstm")(
    dec_emb, initial_state=[initial_h, initial_c]
)

attention_output = AdditiveAttention(name="attention")([decoder_outputs, encoder_outputs])
combined = Concatenate(name="decoder_attention_concat")([decoder_outputs, attention_output])
logits = TimeDistributed(Dense(decoder_vocab_size, activation="softmax"), name="token_classifier")(combined)

model = Model([context_input, question_input, decoder_input], logits, name="attention_seq2seq_qa")
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)
model.summary()

### 2.3 Train for 15 epochs
Early stopping is included but configured to allow the required 15 epochs unless validation loss fails to improve for four consecutive epochs.

**Observation:** Training accuracy normally rises faster than validation accuracy. A widening loss gap indicates overfitting; restoring the best validation weights protects the final model from later degradation.

In [ ]:
# Train for exactly 15 epochs as required by the assignment.
# No EarlyStopping is used because the assignment explicitly requires a minimum of 15–20 epochs.

history = model.fit(
    [ctx_tr, q_tr, dec_in_tr],
    dec_tgt_tr,
    sample_weight=train_sample_weights,
    validation_data=(
        [ctx_val, q_val, dec_in_val],
        dec_tgt_val,
        val_sample_weights
    ), 
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

model.save(OUTPUT_DIR / "attention_seq2seq_qa.keras")

print(f"Completed epochs: {len(history.history['loss'])}")


### 2.4 Plot Training vs Validation Loss and Accuracy

Padding positions are masked using sample weights, so the reported token accuracy reflects answer-token prediction rather than easy prediction of padded zeros.

**Observation:** Training and validation loss/accuracy should be considered together. A large gap between training and validation performance can indicate overfitting.

**Inference:** If validation loss decreases while validation accuracy improves or stabilizes, the model is learning useful answer-token patterns. If training performance continues improving while validation performance degrades, generalization is limited.


In [ ]:
hist = pd.DataFrame(history.history)
display(hist)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(hist["loss"], marker="o", label="Training loss")
axes[0].plot(hist["val_loss"], marker="o", label="Validation loss")
axes[0].set_title("Training vs Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Sparse categorical cross-entropy")
axes[0].legend()

axes[1].plot(hist["accuracy"], marker="o", label="Training accuracy")
axes[1].plot(hist["val_accuracy"], marker="o", label="Validation accuracy")
axes[1].set_title("Training vs Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Token accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_validation_curves.png", dpi=150)
plt.show()

### 2.5 Generate answers and evaluate EM/F1
The decoder uses greedy generation. Exact Match (EM) equals 1 only when prediction and ground truth match after normalization. Token-level F1 measures overlap between generated and reference answer tokens.

**Observation:** Token accuracy can look high because many target positions are padding. EM and token-level F1 are more meaningful QA metrics because they evaluate the generated answer text itself.

In [ ]:
# Decoder vocabulary and special-token IDs.
id_to_word = {idx: word for word, idx in decoder_tokenizer.word_index.items()}

# IMPORTANT: Tokenizer keeps the angle brackets in these special tokens.
start_id = decoder_tokenizer.word_index["<start>"]
end_id = decoder_tokenizer.word_index["<end>"]
oov_id = decoder_tokenizer.word_index.get("<oov>")
pad_id = 0

print("Special token IDs:")
print("<start>:", start_id)
print("<end>:", end_id)
print("<oov>:", oov_id)

def ids_to_answer(token_ids):
    words = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id in (pad_id, start_id, end_id):
            continue
        word = id_to_word.get(token_id, "<oov>")
        if word != "<oov>":
            words.append(word)
    return " ".join(words).strip()

def greedy_decode(context_one, question_one, max_len=answer_maxlen - 1):
    generated = [start_id]

    for _ in range(max_len - 1):
        dec = pad_sequences(
            [generated],
            maxlen=answer_maxlen - 1,
            padding="post",
            truncating="post"
        )

        probabilities = model.predict(
            [context_one[None, :], question_one[None, :], dec],
            verbose=0
        )

        next_position = len(generated) - 1
        next_id = int(np.argmax(probabilities[0, next_position]))

        if next_id in (end_id, pad_id):
            break

        generated.append(next_id)

    return ids_to_answer(generated)

def normalized_tokens(text):
    return clean_text(text).split()

def exact_match(prediction, truth):
    return int(clean_text(prediction) == clean_text(truth))

def token_f1(prediction, truth):
    pred_tokens = normalized_tokens(prediction)
    truth_tokens = normalized_tokens(truth)

    if not pred_tokens and not truth_tokens:
        return 1.0
    if not pred_tokens or not truth_tokens:
        return 0.0

    pred_counts = {}
    truth_counts = {}

    for token in pred_tokens:
        pred_counts[token] = pred_counts.get(token, 0) + 1
    for token in truth_tokens:
        truth_counts[token] = truth_counts.get(token, 0) + 1

    overlap = sum(
        min(count, truth_counts.get(token, 0))
        for token, count in pred_counts.items()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(truth_tokens)
    return 2 * precision * recall / (precision + recall)

EVAL_SAMPLES = min(300, len(val_idx))
eval_indices = val_idx[:EVAL_SAMPLES]

results = []
for idx in eval_indices:
    prediction = greedy_decode(context_pad[idx], question_pad[idx])
    truth = df.iloc[idx]["answer_clean"]

    results.append({
        "row_id": int(idx),
        "question": df.iloc[idx]["question_clean"],
        "true_answer": truth,
        "predicted_answer": prediction,
        "exact_match": exact_match(prediction, truth),
        "token_f1": token_f1(prediction, truth),
        "context_was_truncated": int(df.iloc[idx]["context_len"] > context_maxlen)
    })

results_df = pd.DataFrame(results)

print(f"Validation EM on {EVAL_SAMPLES} examples: {results_df['exact_match'].mean():.4f}")
print(f"Validation token F1 on {EVAL_SAMPLES} examples: {results_df['token_f1'].mean():.4f}")

results_df.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)
display(results_df.head())


### 2.6 Display five sample predictions

**Inference:** Predictions should be inspected qualitatively alongside numerical metrics. A low EM with a higher F1 often means the model generated a partially correct answer but missed words, word order, or answer boundaries.

In [ ]:
sample_predictions = results_df.sample(min(5, len(results_df)), random_state=SEED)
display(sample_predictions[["question", "true_answer", "predicted_answer", "exact_match", "token_f1"]])
sample_predictions.to_csv(OUTPUT_DIR / "five_sample_predictions.csv", index=False)

## Task 3 — Error Analysis and Architectural Refinement

### 3.1 Error Categorization

The following cell selects two incorrect predictions and automatically produces an evidence-based diagnosis using:

1. **Long-context truncation** — whether the original context is longer than the retained cutoff and whether the answer is absent from the retained context prefix.
2. **Decoder OOV** — whether answer words are outside the decoder vocabulary.
3. **Attention/decoding misalignment** — if the answer is retained and decoder-vocabulary coverage is adequate but the generated answer is still incorrect.

The third category is reported as a **likely** cause rather than a proven internal attention failure, because a wrong generated answer alone cannot prove attention misalignment.


In [ ]:
# Select two incorrect predictions.
error_examples = results_df[results_df["exact_match"] == 0].copy()

if len(error_examples) < 2:
    error_examples = results_df.sort_values("token_f1").copy()

error_examples = error_examples.sort_values(
    ["context_was_truncated", "token_f1"],
    ascending=[False, True]
).head(2).copy()

decoder_vocab = set(decoder_tokenizer.word_index.keys())

analysis_rows = []

for n, (_, row) in enumerate(error_examples.iterrows(), start=1):
    original = df.iloc[int(row["row_id"])]

    original_context = original["context_clean"]
    retained_context = " ".join(original_context.split()[:context_maxlen])

    answer_tokens = set(original["answer_clean"].split())
    answer_oov_words = sorted(
        word for word in answer_tokens
        if word not in decoder_vocab
    )

    answer_present_in_retained_context = (
        clean_text(original["answer_clean"]) in clean_text(retained_context)
    )

    if (
        original["context_len"] > context_maxlen
        and not answer_present_in_retained_context
    ):
        diagnosis = "Long-context truncation"
        evidence = (
            "The original context exceeds the cutoff and the normalized "
            "ground-truth answer is not present in the retained context prefix."
        )
    elif answer_oov_words:
        diagnosis = "Out-of-vocabulary (OOV) limitation"
        evidence = (
            "At least one ground-truth answer token is outside the decoder "
            "vocabulary, so the decoder cannot reproduce that word exactly."
        )
    else:
        diagnosis = "Likely attention/decoding misalignment"
        evidence = (
            "The answer is retained in the context and its words are covered "
            "by the decoder vocabulary, but the generated sequence is still "
            "incorrect. This suggests that the decoder did not focus on or "
            "decode the correct evidence span."
        )

    analysis_rows.append({
        "error_number": n,
        "row_id": int(row["row_id"]),
        "question": row["question"],
        "ground_truth": row["true_answer"],
        "prediction": row["predicted_answer"],
        "token_f1": round(float(row["token_f1"]), 4),
        "original_context_length": int(original["context_len"]),
        "context_cutoff": int(context_maxlen),
        "context_truncated": bool(original["context_len"] > context_maxlen),
        "answer_present_in_retained_context": bool(answer_present_in_retained_context),
        "answer_oov_words": ", ".join(answer_oov_words) if answer_oov_words else "None",
        "diagnosis": diagnosis,
        "evidence": evidence
    })

analysis_df = pd.DataFrame(analysis_rows)

display(analysis_df)

analysis_df.to_csv(
    OUTPUT_DIR / "two_error_examples_with_analysis.csv",
    index=False
)

# Print submission-ready written observations and inferences.
for _, row in analysis_df.iterrows():
    print("\n" + "=" * 90)
    print(f"ERROR {int(row['error_number'])}")
    print("=" * 90)
    print(f"Question: {row['question']}")
    print(f"Ground truth: {row['ground_truth']}")
    print(f"Prediction: {row['prediction']}")
    print(f"Token-level F1: {row['token_f1']:.4f}")
    print(
        f"Context length: {row['original_context_length']} tokens; "
        f"cutoff: {row['context_cutoff']} tokens"
    )
    print(f"Context truncated: {row['context_truncated']}")
    print(
        "Answer present in retained context: "
        f"{row['answer_present_in_retained_context']}"
    )
    print(f"Decoder OOV answer words: {row['answer_oov_words']}")
    print(f"Diagnosis: {row['diagnosis']}")
    print(f"Evidence: {row['evidence']}")

print("\nOBSERVATION:")
print(
    "The two selected errors are diagnosed using the actual context length, "
    "retained context, decoder vocabulary coverage, prediction, and F1 score."
)

print("\nINFERENCE:")
print(
    "When the correct evidence is removed by truncation, the model cannot "
    "recover it. When the evidence remains but the answer is still wrong, "
    "decoder/attention alignment becomes the leading explanation. OOV tokens "
    "are a separate vocabulary limitation."
)


### Submission-ready interpretation

The preceding cell prints two actual failed examples after the model is run. Unlike a fixed template, the notebook uses the observed prediction, context length, cutoff, retained-answer check, decoder vocabulary coverage, and token-level F1 to produce the diagnosis.

**Observation:** The selected errors demonstrate concrete failure modes of a generative QA decoder. The diagnosis is evidence-based rather than assumed.

**Inference:** Long-context truncation is especially important because the model cannot attend to tokens removed during preprocessing. If the answer remains in the retained context but generation is incorrect, the failure is more consistent with attention/decoding misalignment. Decoder OOV coverage can independently prevent exact reproduction of rare answer words.


### 3.2 Generative vs Extractive QA

| Aspect | Generative Encoder-Decoder | Extractive Span-Prediction |
|---|---|---|
| Output | Generates answer tokens one at a time | Predicts start and end positions in the given context |
| Strength | Can paraphrase or synthesize answers | Strong evidence grounding for SQuAD-style answers |
| Main risk | Can hallucinate words not supported by context | Cannot answer if the required text is absent from the context |
| Best fit for SQuAD v1.1 | Less naturally aligned with extractive labels | Highly aligned because SQuAD provides answer spans |

**Inference:** Extractive span-prediction is less prone to hallucination for SQuAD-style reading comprehension because it constrains the answer to a span from the supplied context. A generative decoder can produce fluent words that are not present in the evidence.

**Long-context optimization:** Use a **sliding-window encoder with overlap**. Split long contexts into overlapping chunks, run the QA model on each chunk, and select the highest-scoring answer/span. This reduces the probability that an answer near the end of a long passage is lost because of a fixed sequence-length cutoff.

**Observation:** This directly addresses the truncation failure mode identified in Task 3.1.

**Inference:** For long SQuAD contexts, overlapping windows provide a practical improvement without requiring a completely different model architecture.


# Final Assignment Compliance Checklist

Before exporting the notebook to PDF, verify:

- [x] Nested SQuAD JSON parsed.
- [x] Required columns: `title`, `context`, `question`, `answer_text`.
- [x] Lowercasing and whitespace/special-character preprocessing.
- [x] Sentence/sequence length distributions displayed.
- [x] Context, question, and answer word-to-index mappings displayed.
- [x] `pad_sequences` applied.
- [x] Vocabulary sizes and sequence-length cutoffs displayed.
- [x] Context/question Embedding + BiLSTM encoders.
- [x] Recurrent LSTM decoder with Attention.
- [x] Sparse Categorical Cross-Entropy + Adam.
- [x] Exactly 15 training epochs, satisfying the 15–20 epoch requirement.
- [x] Padding masked during training/validation metrics.
- [x] Training vs validation loss and accuracy curves.
- [x] Holdout validation evaluation.
- [x] Exact Match and token-level F1.
- [x] Five predictions with ground-truth answers.
- [x] Two incorrect/truncated predictions selected automatically.
- [x] Evidence-based diagnosis: truncation, OOV, or likely attention/decoding misalignment.
- [x] Generative vs extractive architecture comparison.
- [x] Long-context optimization proposed.
- [x] Observation and inference included throughout.

**Submission:** Run all cells in Jupyter Notebook so that the model metrics, plots, five predictions, and two actual error analyses are visible in the executed notebook, then export the executed notebook as PDF.
